# 11 — Modelo de preço v2 (com duração estimada) e deploy no Model Serving via API

**Objetivo:** prever `vlr_pago` **antes da corrida**, usando a duração estimada como feature, e publicar um endpoint REST consumido pelo app web.

**Mudanças em relação ao KNN da aula (`06_ml`/`07_deploy`):**
1. Calendário calculado a partir do **horário da solicitação** (proxy `entrega − duração`), corrigindo o vazamento temporal.
2. Nova feature `duracao_estimada` (OOF do notebook 09).
3. Um **modelo encadeado** (`mlflow.pyfunc`) recebe apenas coordenadas e data/hora e devolve preço, duração e distância. O app não precisa conhecer as features.
4. O endpoint é criado/atualizado pela **API REST de Serving** do Databricks.

In [ ]:
%run ./00_features_corrida

In [ ]:
import inspect
import mlflow
import sklearn
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

def root_mean_squared_error(y, y_hat):
    return float(np.sqrt(mean_squared_error(y, y_hat)))
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

mlflow.autolog(disable=True)
mlflow.set_registry_uri("databricks-uc")
SEMENTE = 42
NOME_MODELO = "desloc.models.corrida_desloc"
NOME_MODELO_DURACAO = "desloc.models.duracao_corrida"

base = spark.sql("""
    SELECT f.*, o.duracao_estimada
    FROM desloc.gold.corridas_features f
    JOIN desloc.gold.duracao_oof o USING (id_corrida)
    WHERE NOT f.outlier_duracao
""").toPandas()
print(f"Corridas: {len(base)}")

## 1. Comparação justa (mesmas 5 dobras)

| Cenário | Features |
|---|---|
| `knn_aula` | KNN de 5 vizinhos da aula, com calendário da **entrega** e coordenadas (vazamento, apenas referência) |
| `sem_duracao` | distância + calendário da solicitação |
| `com_duracao_estimada` | idem + `duracao_estimada` (OOF), como pede o enunciado |
| `com_duracao_real` | idem + duração real: **teto teórico**, não disponível na cotação |

In [ ]:
base_aula = base.assign(
    hora_entrega=pd.to_datetime(base.datahora_entregue).dt.hour,
    dia_entrega=pd.to_datetime(base.datahora_entregue).dt.day,
    mes_entrega=pd.to_datetime(base.datahora_entregue).dt.month,
    dia_semana_entrega=((pd.to_datetime(base.datahora_entregue).dt.dayofweek + 1) % 7 + 1))
FEATURES_AULA = ["dia_entrega", "dia_semana_entrega", "mes_entrega", "hora_entrega", "distancia_lalo",
                 "latitude_origem", "longitude_origem", "latitude_destino", "longitude_destino"]

cv = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)
y = base["vlr_pago"]

def candidatos():
    return {
        "media_dummy": DummyRegressor(strategy="mean"),
        "regressao_linear": Pipeline([("escala", StandardScaler()), ("modelo", LinearRegression())]),
        "knn_5": Pipeline([("escala", MinMaxScaler()), ("modelo", KNeighborsRegressor(n_neighbors=5))]),
        "random_forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEMENTE),
        "gradient_boosting": GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05,
                                                       random_state=SEMENTE),
    }

def avaliar(cenario, nome, modelo, X):
    r = cross_validate(modelo, X, y, cv=cv, scoring=("neg_mean_absolute_error", "neg_root_mean_squared_error"))
    return {"cenario": cenario, "modelo": nome, "MAE_R$": -r["test_neg_mean_absolute_error"].mean(),
            "RMSE_R$": -r["test_neg_root_mean_squared_error"].mean()}

linhas = [avaliar("knn_aula", "knn_5", candidatos()["knn_5"], base_aula[FEATURES_AULA])]
cenarios = {"sem_duracao": FEATURES_DURACAO,
            "com_duracao_estimada": FEATURES_PRECO,
            "com_duracao_real": FEATURES_DURACAO + ["duracao"]}
for cenario, colunas in cenarios.items():
    for nome, modelo in candidatos().items():
        linhas.append(avaliar(cenario, nome, modelo, base[colunas]))
comparacao = pd.DataFrame(linhas)
display(comparacao.pivot(index="modelo", columns="cenario", values="MAE_R$").round(3))

## 2. Escolha do modelo de preço

O enunciado pede que a duração estimada seja feature do preço. Por isso o modelo final é o de menor MAE no cenário `com_duracao_estimada`. Também é reportado um **teste reservado (20%)**, avaliado uma única vez.

In [ ]:
com_duracao = comparacao[(comparacao.cenario == "com_duracao_estimada") & (comparacao.modelo != "media_dummy")]
MELHOR_PRECO = com_duracao.sort_values("MAE_R$").modelo.iloc[0]
cv_mae = com_duracao.set_index("modelo").loc[MELHOR_PRECO, "MAE_R$"]
cv_rmse = com_duracao.set_index("modelo").loc[MELHOR_PRECO, "RMSE_R$"]
cv_mae_aula = comparacao[comparacao.cenario == "knn_aula"]["MAE_R$"].iloc[0]
cv_mae_sem = comparacao[(comparacao.cenario == "sem_duracao") & (comparacao.modelo == MELHOR_PRECO)]["MAE_R$"].iloc[0]
print(f"Modelo de preço: {MELHOR_PRECO} | MAE CV R$ {cv_mae:.2f} (sem duração: R$ {cv_mae_sem:.2f}; "
      f"KNN da aula: R$ {cv_mae_aula:.2f})")

X_tr, X_te, y_tr, y_te = train_test_split(base[FEATURES_PRECO], y, test_size=0.2, random_state=SEMENTE)
holdout = candidatos()[MELHOR_PRECO].fit(X_tr, y_tr)
pred_te = holdout.predict(X_te)
teste_mae, teste_rmse = mean_absolute_error(y_te, pred_te), root_mean_squared_error(y_te, pred_te)
print(f"Teste reservado ({len(y_te)} corridas): MAE R$ {teste_mae:.2f} | RMSE R$ {teste_rmse:.2f}")
print(f"Total real R$ {y_te.sum():.2f} x previsto R$ {pred_te.sum():.2f}")

## 3. Modelo encadeado para o Serving

O `CorridaDesloc` recebe `latitude_origem, longitude_origem, latitude_destino, longitude_destino, datahora_solicitacao`. Ele calcula as features com `preparar_features`, prevê a duração, usa a duração como feature do preço e devolve `valor_previsto`, `duracao_estimada_min` e `distancia_km`.

Na inferência, o modelo de duração é treinado com **todas** as corridas válidas, enquanto o modelo de preço é treinado com as **OOF**, prática padrão de *stacking*.

In [ ]:
from sklearn.base import clone

usuario = spark.sql("SELECT current_user()").first()[0]
versao_duracao = max(MlflowClient().search_model_versions(f"name = '{NOME_MODELO_DURACAO}'"),
                     key=lambda v: int(v.version))
modelo_duracao = mlflow.sklearn.load_model(f"models:/{NOME_MODELO_DURACAO}/{versao_duracao.version}")
modelo_preco = clone(candidatos()[MELHOR_PRECO]).fit(base[FEATURES_PRECO], y)


class CorridaDesloc(mlflow.pyfunc.PythonModel):
    def __init__(self, modelo_duracao, modelo_preco):
        self.modelo_duracao = modelo_duracao
        self.modelo_preco = modelo_preco

    def predict(self, context, model_input, params=None):
        entrada = pd.DataFrame(model_input)
        features = preparar_features(entrada)
        features["duracao_estimada"] = self.modelo_duracao.predict(features[FEATURES_DURACAO])
        preco = self.modelo_preco.predict(features[FEATURES_PRECO])
        return pd.DataFrame({
            "valor_previsto": np.round(np.maximum(preco, 0), 2),
            "duracao_estimada_min": np.round(np.maximum(features["duracao_estimada"], 1), 1),
            "distancia_km": np.round(features["distancia_lalo"], 3),
        })


modelo_corrida = CorridaDesloc(modelo_duracao, modelo_preco)
exemplo = pd.DataFrame([
    {"latitude_origem": -23.5614, "longitude_origem": -46.6559, "latitude_destino": -23.5874,
     "longitude_destino": -46.6576, "datahora_solicitacao": "2026-11-02 08:15:00"},
    {"latitude_origem": -23.5505, "longitude_origem": -46.6333, "latitude_destino": -23.5200,
     "longitude_destino": -46.6000, "datahora_solicitacao": "2026-11-06 19:30:00"},
])
display(pd.concat([exemplo, modelo_corrida.predict(None, exemplo)], axis=1))

In [ ]:
import numpy, pandas, cloudpickle
mlflow.set_experiment(f"/Users/{usuario}/desloc-corrida")
requisitos = [f"mlflow=={mlflow.__version__}", f"scikit-learn=={sklearn.__version__}",
              f"pandas=={pandas.__version__}", f"numpy=={numpy.__version__}",
              f"cloudpickle=={cloudpickle.__version__}"]

with mlflow.start_run(run_name=f"corrida-{MELHOR_PRECO}") as run:
    mlflow.log_params({"modelo_preco": MELHOR_PRECO, "features_preco": ",".join(FEATURES_PRECO),
                       "modelo_duracao": f"{NOME_MODELO_DURACAO}/{versao_duracao.version}",
                       "corridas_treino": len(base)})
    mlflow.log_metrics({"cv_mae": cv_mae, "cv_rmse": cv_rmse, "cv_mae_sem_duracao": cv_mae_sem,
                        "cv_mae_knn_aula": cv_mae_aula, "teste_mae": teste_mae, "teste_rmse": teste_rmse})
    mlflow.log_table(comparacao, "comparacao_cenarios.json")
    argumentos = {"python_model": modelo_corrida,
                  "signature": infer_signature(exemplo, modelo_corrida.predict(None, exemplo)),
                  "input_example": exemplo.head(1),
                  "pip_requirements": requisitos,
                  "registered_model_name": NOME_MODELO}
    parametro = "name" if "name" in inspect.signature(mlflow.pyfunc.log_model).parameters else "artifact_path"
    argumentos[parametro] = "model"
    mlflow.pyfunc.log_model(**argumentos)
    RUN_ID = run.info.run_id

versoes = [v for v in MlflowClient().search_model_versions(f"name = '{NOME_MODELO}'") if v.run_id == RUN_ID]
VERSAO_MODELO = str(versoes[0].version)
carregado = mlflow.pyfunc.load_model(f"models:/{NOME_MODELO}/{VERSAO_MODELO}")
display(carregado.predict(exemplo))
print(f"Registrado: {NOME_MODELO} versão {VERSAO_MODELO} (run {RUN_ID})")
dbutils.jobs.taskValues.set(key="versao_modelo", value=VERSAO_MODELO)

## 4. Model Serving via API REST

O endpoint é criado ou atualizado pela **API REST** do Databricks, usando a identidade do notebook (`WorkspaceClient().api_client`, sem token no código):

| Operação | Chamada |
|---|---|
| Criar | `POST /api/2.0/serving-endpoints` |
| Nova versão (retreino) | `PUT /api/2.0/serving-endpoints/{nome}/config` |
| Status | `GET /api/2.0/serving-endpoints/{nome}` |
| Inferência | `POST /serving-endpoints/{nome}/invocations` |

O mesmo fluxo existe fora do Databricks em `scripts/serving_endpoint.py` (repositório Git), que foi o usado para publicar o endpoint do app. A célula começa desativada para não consumir cota sem querer.

In [ ]:
PUBLICAR_ENDPOINT = False
NOME_ENDPOINT = "desloc-corrida"

if PUBLICAR_ENDPOINT:
    import time
    from databricks.sdk import WorkspaceClient
    api = WorkspaceClient().api_client
    entidade = {"name": "corrida-desloc", "entity_name": NOME_MODELO, "entity_version": VERSAO_MODELO,
                "workload_size": "Small", "scale_to_zero_enabled": True}
    existentes = {e["name"] for e in api.do("GET", "/api/2.0/serving-endpoints").get("endpoints", [])}
    if NOME_ENDPOINT in existentes:
        api.do("PUT", f"/api/2.0/serving-endpoints/{NOME_ENDPOINT}/config", body={"served_entities": [entidade]})
    else:
        api.do("POST", "/api/2.0/serving-endpoints",
               body={"name": NOME_ENDPOINT, "config": {"served_entities": [entidade]}})
    for _ in range(90):
        estado = api.do("GET", f"/api/2.0/serving-endpoints/{NOME_ENDPOINT}")["state"]
        if estado.get("ready") == "READY" and estado.get("config_update") == "NOT_UPDATING":
            break
        time.sleep(20)
    print(estado)
    resposta = api.do("POST", f"/serving-endpoints/{NOME_ENDPOINT}/invocations",
                      body={"dataframe_records": exemplo.to_dict(orient="records")})
    print(resposta)
else:
    print("Publicação desativada nesta execução — use scripts/serving_endpoint.py ou ative PUBLICAR_ENDPOINT.")

## 5. Interpretação

- **Vazamento corrigido:** o calendário vem do horário da solicitação, então o modelo pode ser usado na cotação, antes da corrida.
- **Duração estimada:** entra no preço como pede o enunciado. Como a duração estimada é derivada das mesmas features (distância e horário), o ganho sobre o modelo sem duração é pequeno. A duração **real** melhora o preço, o que mostra que um modelo de duração melhor (rotas e trânsito) é o caminho para ganhar precisão.
- **Comparação com a aula:** o KNN original tem erro maior mesmo usando informações da entrega.
- **Monitoramento:** o backend grava cada previsão (entrada, versão, saída) em JSONL. Com o `vlr_pago` real, isso permite medir o erro em produção e drift, as pendências do notebook 08.